Assignment 04

In [1]:
# Importing Libraries
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

np.random.seed(42)
tf.random.set_seed(42)


In [2]:
# Dataset
samples = 5000
length = 400
channels = 4
classes = 3

X = np.random.rand(samples, length, channels).astype("float32")
y = np.random.randint(0, classes, size=(samples,))

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [3]:
# SpliceFinder CNN Model
def cnn_model():
    model = Sequential([
        Conv1D(
            filters=50,
            kernel_size=9,
            activation="relu",
            input_shape=(400, 4)
        ),
        Flatten(),
        Dense(100, activation="relu"),
        Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [4]:
# Early Stopping (Patience = 3)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


In [5]:
# Training CPU
tf.keras.backend.clear_session()

with tf.device("/CPU:0"):
    cpu_model = cnn_model()

    start_time = time.time()
    cpu_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )
    cpu_time = time.time() - start_time


Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.3507 - loss: 1.2667 - val_accuracy: 0.3270 - val_loss: 1.0998
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.4044 - loss: 1.0834 - val_accuracy: 0.3330 - val_loss: 1.1036
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.4648 - loss: 1.0376 - val_accuracy: 0.3360 - val_loss: 1.1069
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.5510 - loss: 0.9600 - val_accuracy: 0.3370 - val_loss: 1.1322


In [6]:
# Training GPU
tf.keras.backend.clear_session()

gpu_time = None

if tf.config.list_physical_devices("GPU"):
    with tf.device("/GPU:0"):
        gpu_model = cnn_model()

        start_time = time.time()
        gpu_model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=32,
            callbacks=[early_stop],
            verbose=1
        )
        gpu_time = time.time() - start_time
else:
    print("No GPU detected.")


Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.3371 - loss: 1.2046 - val_accuracy: 0.3270 - val_loss: 1.0991
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4571 - loss: 1.0707 - val_accuracy: 0.3270 - val_loss: 1.1219
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5871 - loss: 0.9487 - val_accuracy: 0.3070 - val_loss: 1.1708
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7107 - loss: 0.7342 - val_accuracy: 0.3150 - val_loss: 1.2487


In [7]:
# Results
print("\nTraining Time Comparison")
print(f"CPU Time: {cpu_time:.2f} seconds")

if gpu_time is not None:
    print(f"GPU Time: {gpu_time:.2f} seconds")
    print(f"Faster By: {cpu_time / gpu_time:.2f}×")



Training Time Comparison
CPU Time: 15.83 seconds
GPU Time: 5.59 seconds
Faster By: 2.83×
